# TMEW-1 / HMWMC v2 — Colab and Jupyter runner

Runs the repaired v2 curriculum (`tmew1_run_v2.py`) at scale: multi-seed full runs with the
zero-shot emergent-capability probes, aggregated across seeds, plus the Phase-1 branch
experiments (`tmew1_branch_runner_v2.py`).

**Before running on Colab:** *Runtime → Change runtime type → GPU* (T4 or better). A full
4-epoch curriculum takes about 25 min on an RTX 3070 and roughly 45–60 min on a T4; the
checkpoint for each tier is about 150 MB.

Outputs for every run go to `OUT_ROOT/<run_name>/`:

- `val_tier{N}_ep{E}.json` — validation metrics per tier and epoch;
- `diagnostics_tier{N}.json` — handoff-bucket recall, belief revision, and the emergent probes;
- `checkpoint_tier{N}.pt`.

Mount Google Drive in section 2 so these survive the Colab session.

## 1. Clone repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/ai-wes/hmwmc.git"
REPO_DIR = "hmwmc"
BRANCH = "codex/research-release-readme-ci"   # carries the 2026-11-03 fixes; switch to "main" once merged

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR}/ already exists — pulling latest...")
    !cd {REPO_DIR} && git fetch -q && git checkout -q {BRANCH} && git pull -q
else:
    !git clone -q --branch {BRANCH} {REPO_URL}

os.chdir(REPO_DIR)
!git log --oneline -1
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Colab ships PyTorch with CUDA; this only fills in the small extras.
!pip install -q -r requirements.txt

## 2. Verify CUDA and choose where outputs go

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device  : {torch.cuda.get_device_name(0)}")
    print(f"Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → GPU.")

In [ ]:
# Mount Drive on Colab so runs persist; anywhere else, fall back to a local folder.
import os

USE_DRIVE = True
OUT_ROOT = "runs"
if USE_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        # Blocks until you approve the Google Drive prompt; gives up after
        # 3 minutes so an unattended run still proceeds with local storage.
        drive.mount("/content/drive", timeout_ms=180000)
        OUT_ROOT = "/content/drive/MyDrive/hmwmc_runs"
    except ImportError:
        print("Not on Colab — writing runs to ./runs")
    except Exception as exc:  # noqa: BLE001 — mount declined or timed out
        print(f"Drive mount failed ({exc.__class__.__name__}); writing runs to ./runs instead")
os.makedirs(OUT_ROOT, exist_ok=True)
print("OUT_ROOT =", OUT_ROOT)

### Job runner

`run_jobs` launches training subprocesses with up to `PARALLEL_RUNS` running at once. A default-size run needs
about 3 GB of GPU memory, so `"auto"` allows one run per 4 GB of VRAM (1 on a T4, 5 on an L4, 9 on a 40 GB A100).
Each run logs to `<out_dir>/run.log`; key lines (model shape, Tier 3 validation, probes, any non-finite step) are
echoed here when the run finishes. Runs whose `diagnostics_tier3.json` already exists are skipped, so re-running
a cell resumes after a session reset.

In [ ]:
import os, subprocess, sys, time
import torch

PARALLEL_RUNS = "auto"   # or an integer, e.g. 4

def _parallel_slots():
    if PARALLEL_RUNS != "auto":
        return max(1, int(PARALLEL_RUNS))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    return max(1, int(total_gb // 4))

KEY_LINE_MARKERS = ("model:", "[val] t3", "qacc/who_holds_token", "[probes] t3", "Non-finite", "promot", "exit=")

def run_jobs(jobs, parallel=None):
    """jobs: list of (name, out_dir, cmd). Runs up to `parallel` at once; returns {name: returncode}."""
    parallel = parallel or _parallel_slots()
    pending = []
    for name, out_dir, cmd in jobs:
        if os.path.exists(os.path.join(out_dir, "diagnostics_tier3.json")):
            print(f"{name}: already complete, skipping")
            continue
        os.makedirs(out_dir, exist_ok=True)
        pending.append((name, out_dir, cmd))
    print(f"{len(pending)} job(s) to run, up to {parallel} in parallel")
    running, results, t_start = {}, {}, {}
    while pending or running:
        while pending and len(running) < parallel:
            name, out_dir, cmd = pending.pop(0)
            log = open(os.path.join(out_dir, "run.log"), "w")
            proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, text=True)
            running[name] = (proc, log, out_dir)
            t_start[name] = time.time()
            print(f"[{time.strftime('%H:%M:%S')}] started {name}  ({' '.join(cmd)})")
        for name in list(running):
            proc, log, out_dir = running[name]
            rc = proc.poll()
            if rc is None:
                continue
            log.close()
            del running[name]
            results[name] = rc
            mins = (time.time() - t_start[name]) / 60
            print(f"\n[{time.strftime('%H:%M:%S')}] finished {name} rc={rc} in {mins:.1f} min")
            with open(os.path.join(out_dir, "run.log"), errors="replace") as f:
                for line in f:
                    if any(m in line for m in KEY_LINE_MARKERS):
                        print("   ", line.rstrip()[:160])
        time.sleep(15)
    return results

print("parallel slots:", _parallel_slots())

## 3. Smoke test (about 30 s)

Checks execution only; its metrics are not results. The CI check also fails on any `nan`/`inf` or skipped-backward signal.

In [ ]:
!python tmew1_run_v2.py --smoke --out-dir {OUT_ROOT}/smoke 2>&1 | grep -vE "UserWarning|return inner|^sys:1" | tail -40

## 4. Multi-seed full curriculum with emergent probes

One sequential run per seed (the model needs ~3 GB of GPU memory, so parallel seeds on a T4 are not worth it).
Edit `SEEDS`, `EPOCHS`, `TRAIN_EPISODES`, and `BATCH_SIZE` to scale up. Each run prints the per-tier validation
snapshot and the diagnostics block, including the probes:

- HPM slot specialization (>0.3 interesting)
- causal thread separation (>0.1)
- retroactive cause binding (>0.05)
- state reactivation / replay (>0)
- event boundary chunking and surprise–event correlation (>1.5)

`MODEL_SIZE` picks a network preset: `default` (~12M params), `large` (d_model 384, 8 layers, ~34M, controller retuned),
`xl` (d_model 512, 10 layers, ~72M). `PROBE_EVERY_EPOCH=True` runs the probes after every validation pass and writes
`probes_tier{N}_ep{E}.json`, so section 5 can show how each property evolves over training.

In [ ]:
SEEDS = [0, 1, 2]
EPOCHS = 4
TRAIN_EPISODES = 2048
BATCH_SIZE = 32
MODEL_SIZE = "large"       # "default" | "large" | "xl"  (see tmew1_run_v2.py --help for per-field overrides)
PROBE_EVERY_EPOCH = True   # emergent-property monitor after every validation pass
RUN_PREFIX = f"v2_{MODEL_SIZE}"   # run directories become OUT_ROOT/v2_large_seed{seed}

jobs = []
for seed in SEEDS:
    out_dir = os.path.join(OUT_ROOT, f"{RUN_PREFIX}_seed{seed}")
    cmd = [sys.executable, "-u", "tmew1_run_v2.py",
           "--seed", str(seed), "--epochs", str(EPOCHS),
           "--train-episodes", str(TRAIN_EPISODES), "--batch-size", str(BATCH_SIZE),
           "--size", MODEL_SIZE, "--out-dir", out_dir]
    if PROBE_EVERY_EPOCH:
        cmd.append("--probe-every-epoch")
    jobs.append((f"{RUN_PREFIX}_seed{seed}", out_dir, cmd))
# The large preset needs ~6.5 GB at batch 16 and more at batch 32: allow fewer parallel slots for it.
run_jobs(jobs, parallel=None if MODEL_SIZE == "default" else max(1, _parallel_slots() // 2))

## 5. Aggregate across seeds

Mean and standard deviation (ddof=1) over completed seeds for the final-epoch Tier 3 validation metrics, the holder-query handoff buckets, and the emergent probes, followed by the per-epoch probe trajectory.

In [ ]:
!python tmew1_aggregate_runs.py "{OUT_ROOT}/{RUN_PREFIX}_seed*" --tier 3 --trajectory --json {OUT_ROOT}/{RUN_PREFIX}_summary_tier3.json

In [ ]:
# Plot the probe trajectory (mean ± std across seeds) per tier/epoch.
import json, os
import matplotlib.pyplot as plt

summary = json.load(open(os.path.join(OUT_ROOT, f"{RUN_PREFIX}_summary_tier3.json")))
traj = summary.get("trajectory", {})
if not traj:
    print("no per-epoch probes recorded; set PROBE_EVERY_EPOCH = True in section 4")
else:
    probes = sorted({k for point in traj.values() for k in point})
    points = list(traj)
    fig, axes = plt.subplots(1, len(probes), figsize=(4 * len(probes), 3.2), squeeze=False)
    for ax, p in zip(axes[0], probes):
        means = [traj[pt][p]["mean"] if p in traj[pt] else float("nan") for pt in points]
        stds = [traj[pt][p]["std"] if p in traj[pt] else 0.0 for pt in points]
        ax.errorbar(range(len(points)), means, yerr=stds, marker="o", capsize=3)
        ax.set_title(p, fontsize=9)
        ax.set_xticks(range(len(points)))
        ax.set_xticklabels(points, rotation=90, fontsize=7)
    plt.tight_layout()
    plt.show()

## 5b. Memory ablations on the honest benchmark

Which memory module is load-bearing? Each run removes one path on the default model and curriculum
(`--ablate hpm`: HPM off; `episodic`: episodic bank off; `structured`: state table, typed events and
checkpoints off; `none`: control). Three seeds each is 12 runs; with `PARALLEL_RUNS` set in the job-runner cell they share the GPU
(about 3 GB each), and completed runs are skipped on re-run.

In [ ]:
ABLATIONS = ["none", "hpm", "episodic", "structured"]
ABL_SEEDS = [0, 1, 2]
ABL_EPOCHS = 4
ABL_TRAIN_EPISODES = 2048
ABL_BATCH_SIZE = 32

jobs = []
for ablate in ABLATIONS:
    for seed in ABL_SEEDS:
        out_dir = os.path.join(OUT_ROOT, f"abl_{ablate}_seed{seed}")
        cmd = [sys.executable, "-u", "tmew1_run_v2.py", "--ablate", ablate, "--size", "default",
               "--seed", str(seed), "--epochs", str(ABL_EPOCHS),
               "--train-episodes", str(ABL_TRAIN_EPISODES), "--batch-size", str(ABL_BATCH_SIZE),
               "--probe-every-epoch", "--out-dir", out_dir]
        jobs.append((f"abl_{ablate}_seed{seed}", out_dir, cmd))
run_jobs(jobs)   # uses PARALLEL_RUNS from the job-runner cell

In [ ]:
# Compare ablations: Tier 3 validation and probes, mean ± std across seeds.
for ablate in ABLATIONS:
    print(f"\n################ ablate={ablate} ################")
    !python tmew1_aggregate_runs.py "{OUT_ROOT}/abl_{ablate}_seed*" --tier 3 --json {OUT_ROOT}/abl_{ablate}_summary.json

In [ ]:
# Side-by-side holder query / first-tagged / probes per ablation.
import json, os
rows = []
for ablate in ABLATIONS:
    p = os.path.join(OUT_ROOT, f"abl_{ablate}_summary.json")
    if not os.path.exists(p):
        continue
    s = json.load(open(p))["summary"]
    v, e = s["validation"], s["emergent_probes"]
    def g(d, k): 
        return f"{d[k]['mean']:.3f}±{d[k]['std']:.3f} (n={d[k]['n_seeds']})" if k in d else "—"
    rows.append((ablate, g(v, "qacc/who_holds_token"), g(v, "qacc/who_was_first_tagged"), g(v, "holder_acc"),
                 g(e, "slot_specialization"), g(e, "thread_separation"), g(e, "retroactive_binding")))
print(f"{'ablate':<11}{'who_holds_token':<24}{'first_tagged':<24}{'holder_acc':<24}{'slot_spec':<24}{'thread_sep':<24}{'retro_bind':<24}")
for r in rows:
    print("".join(f"{c:<24}" if i else f"{c:<11}" for i, c in enumerate(r)))

## 6. Larger configurations

Network size is set in section 4 (`MODEL_SIZE`, or `--d-model/--num-layers/--memory-slots/--episodic-slots` for
per-field control). For harder worlds, longer horizons, or HPM variants use the branch runner below, which composes
`WorldConfig`/tier/HPM overrides on top of the same `run_curriculum`. Example: longer Tier 3 horizon and more epochs.

In [ ]:
# Example: Tier 3 at T=96 with 6 epochs, seed 0. Add --seed N to repeat across seeds.
!python tmew1_branch_runner_v2.py --branch baseline \
    --tier3-episode-length 96 --epochs 6 --seed 0 \
    --out-dir {OUT_ROOT}/baseline_T96_seed0 2>&1 | grep -vE "UserWarning|return inner|^sys:1" | tail -60

## 7. Phase-1 branch experiments

Each branch changes one axis (A: harder worlds, B: harder queries, C: HPM architecture, D/E: memory and
stability variants) and is judged against a frozen baseline record by its promotion rubric
(`tmew1_experiments_v2.py`). Run the baseline first; it writes `val.json`, which the other branches read.
Branches run sequentially here because of GPU memory.

Exit codes: 0 promote · 1 kill · 2 undecided · 3 missing capabilities · 4 crash.

In [ ]:
# Step 1: frozen baseline reference (full curriculum from scratch; add --resume <checkpoint_tier2.pt> to shortcut)
BASELINE_DIR = os.path.join(OUT_ROOT, "experiments", "baseline")
!python tmew1_branch_runner_v2.py --branch baseline --out-dir {BASELINE_DIR} 2>&1 | grep -vE "UserWarning|return inner|^sys:1" | tail -40

In [ ]:
# Step 2: branches, sequentially
import subprocess, sys, os
from tmew1_experiments_v2 import BRANCH_IDS

BRANCHES = ["A1", "A2", "C1", "C2"]      # any of BRANCH_IDS; B-branches need --ack-missing-capabilities
EXTRA_ARGS = []                          # e.g. ["--seed", "1"]
BASELINE_RECORD = os.path.join(BASELINE_DIR, "val.json")
assert os.path.exists(BASELINE_RECORD), "run the baseline cell first"

for branch in BRANCHES:
    assert branch in BRANCH_IDS, branch
    out_dir = os.path.join(OUT_ROOT, "experiments", branch)
    os.makedirs(out_dir, exist_ok=True)
    cmd = [sys.executable, "tmew1_branch_runner_v2.py", "--branch", branch,
           "--baseline-record", BASELINE_RECORD, "--out-dir", out_dir, *EXTRA_ARGS]
    if branch.startswith("B"):
        cmd.append("--ack-missing-capabilities")
    print(f"\n=== {branch} ===")
    with open(os.path.join(out_dir, "run.log"), "w") as log:
        rc = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT)
    status = {0: "promote", 1: "kill", 2: "undecided", 3: "missing capabilities", 4: "crash"}.get(rc, f"code={rc}")
    print(f"{branch} → {status}   (log: {out_dir}/run.log)")

In [ ]:
# Step 3: inspect verdicts
import json, glob, os

print(f"{'Branch':<12} {'Verdict':<12} {'Target metric':<34} {'Value':>8}")
print("─" * 70)
for d in sorted(glob.glob(os.path.join(OUT_ROOT, "experiments", "*/"))):
    branch = os.path.basename(d.rstrip("/"))
    verdict_files = sorted(glob.glob(os.path.join(d, "verdict_*.jsonl")))
    if not verdict_files:
        print(f"{branch:<12} {'no verdict':<12}")
        continue
    with open(verdict_files[-1]) as f:
        lines = [json.loads(l) for l in f if l.strip()]
    rec = lines[-1] if lines else {}
    v = rec.get("verdict", "?")
    target = (rec.get("config") or {}).get("rubric", {}).get("target_metric", "—")
    val = (rec.get("val_record") or {}).get(target, float("nan")) if target != "—" else float("nan")
    print(f"{branch:<12} {v:<12} {target:<34} {val:>8.4f}")